# FedFairGNN — Large-scale scalability study (ogbn-products, 2.4M nodes)

Run this on a **GPU runtime** (Colab: Runtime → Change runtime type → GPU; Kaggle: enable GPU).
It installs dependencies, pulls the repo, and runs the FedFairGNN scalability
experiment on **ogbn-products (2.4M nodes / 61M edges)** with neighbor sampling.
When it finishes, download `results/summary.jsonl` and send it back — the
paper's tables/figures ingest it automatically.

*Note:* ogbn-products is a **scalability stress-test**; its label and sensitive
attribute are documented structural proxies (see `src/data/datasets.py`), not a
demographic-fairness benchmark.

In [ ]:
import torch, sys
v = torch.__version__.split('+')[0]
cu = 'cu121' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| wheels for', v, cu)
!pip -q install torch_geometric ogb
!pip -q install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-{v}+{cu}.html
print('deps installed; CUDA available:', torch.cuda.is_available())

In [ ]:
import os
if not os.path.isdir('FedFairGNN'):
    !git clone --depth 1 https://github.com/vinhqdang/FedFairGNN.git
%cd FedFairGNN

In [ ]:
# Runs 7 methods on ogbn-products with neighbor sampling on the GPU.
# First run downloads ogbn-products (~1.4 GB). Expect tens of minutes on GPU.
!python -m experiments.run_large_scale

In [ ]:
# Inspect and package results to send back.
import json
rows = [json.loads(l) for l in open('results/summary.jsonl')]
ogbn = [r for r in rows if r.get('dataset') == 'ogbn_products']
for r in ogbn:
    print(f"{r['exp_name']:16s} AUC={r.get('final_auc'):.3f} "
          f"DPD={r.get('final_dpd'):.3f} EOD={r.get('final_eod'):.3f}")
!zip -j fedfairgnn_ogbn_results.zip results/summary.jsonl
print('\nDownload fedfairgnn_ogbn_results.zip and send it back.')